In [ ]:
#!/usr/bin/env python3

import os
import sys

# --- Configuration ---
# Set the path to the directory containing your FASTA files.
FASTA_FILES_DIR = "C:\\Users\\rosie\\OneDrive\\Desktop\\Demo-Analysis\\data"


# --- REVISED FUNCTION TO SUPPORT MULTI-FASTA FILES ---
def load_sequences_from_files(filenames: list[str], input_dir: str) -> dict[str, str]:
    
    print(f"--- Loading FASTA data from local directory: '{input_dir}' ---")
    
    if not os.path.isdir(input_dir):
        print(f"\n[ERROR] The directory '{input_dir}' was not found.", file=sys.stderr)
        return {}
        
    loaded_sequences = {}
    for filename in filenames:
        filepath = os.path.join(input_dir, filename)
        
        try:
            with open(filepath, 'r') as f:
                current_id = None
                current_sequence_parts = []
                
                for line in f:
                    line = line.strip()
                    if not line:
                        continue

                    if line.startswith('>'):
                        if current_id and current_sequence_parts:
                            sequence = "".join(current_sequence_parts)
                            loaded_sequences[current_id] = sequence
                            print(f"  [SUCCESS] Loaded sequence for {current_id} from '{filename}'")
                        
                        try:
                            current_id = line.split('|')[1]
                        except IndexError:
                            print(f"  [WARNING] Could not parse UniProt ID from header '{line}' in '{filename}'. Skipping entry.")
                            current_id = None
                        
                        current_sequence_parts = []
                    
                    elif current_id:
                        current_sequence_parts.append(line)

                if current_id and current_sequence_parts:
                    sequence = "".join(current_sequence_parts)
                    loaded_sequences[current_id] = sequence
                    print(f"  [SUCCESS] Loaded sequence for {current_id} from '{filename}'")

        except FileNotFoundError:
            print(f"  [INFO] No file found named '{filename}'")
            
    return loaded_sequences


def main():
    """
    Main function to automatically find and load all FASTA files from a directory.
    """
    if not os.path.isdir(FASTA_FILES_DIR):
        print(f"\n[ERROR] The directory '{FASTA_FILES_DIR}' was not found.", file=sys.stderr)
        print("Please check the FASTA_FILES_DIR path at the top of the script.", file=sys.stderr)
        return {}

    print(f"--- Scanning directory for FASTA files: '{FASTA_FILES_DIR}' ---")
    try:
        fasta_files_to_query = [
            f for f in os.listdir(FASTA_FILES_DIR) 
            if f.lower().endswith(('.fasta', '.fa'))
        ]
    except OSError as e:
        print(f"\n[ERROR] Could not read the directory '{FASTA_FILES_DIR}': {e}", file=sys.stderr)
        return {}
    
    if not fasta_files_to_query:
        print("\n[INFO] No FASTA (.fasta or .fa) files were found in the specified directory.", file=sys.stderr)
        return {}
        
    print(f"--- Found {len(fasta_files_to_query)} FASTA files to process. ---")

    results = load_sequences_from_files(fasta_files_to_query, FASTA_FILES_DIR)
    
    if not results:
        print("\nCould not load any data, despite finding files. Please check file contents and permissions.", file=sys.stderr)
        return {}

    print("\n\n--- Final Results (Sample) ---")
    for accession_id, sequence in list(results.items())[:5]:
        print(f"\nAccession ID: {accession_id}")
        print(f"  Sequence (first 60 chars):\n---\n{sequence[:60]}...")
            
    print("\n--- Process Complete ---")
    return results

# ============================ HOW TO USE ============================
# 1. SET the `FASTA_FILES_DIR` variable at the top of this script.
# 2. RUN the script. 
# 3. Two variables will be available for you to use after it runs:
#    - `fasta_results`: A dictionary mapping {Accession_ID -> Sequence}.
#    - `sequence_list`: A list of all sequence strings.
# ==================================================================

if __name__ == "__main__":
    # The main function returns the dictionary of results
    fasta_results = main()

    # Create the list of sequences as a separate variable
    sequence_list = []

    if fasta_results:
        # --- NEW: Create a list containing only the sequence strings ---
        sequence_list = list(fasta_results.values())
        
        print(f"\nSuccessfully loaded {len(fasta_results)} sequences.")
        print("You can now work with two variables:")
        print(f"  1. 'fasta_results' (a dictionary of ID->sequence, size: {len(fasta_results)})")
        print(f"  2. 'sequence_list' (a list of all sequences, size: {len(sequence_list)})")

        
        # --- Example 1: Using the dictionary for lookup ---
        print("\n--- Example: Accessing a sequence by ID from the DICTIONARY ---")
        test_id = "P0A7B8" 
        if test_id in fasta_results:
            print(f"  Sequence length for '{test_id}' is: {len(fasta_results[test_id])} amino acids.")
        else:
            print(f"  Test ID '{test_id}' was not found in the loaded results.")

        # --- Example 2: Using the list for iteration/indexing ---
        if sequence_list:
            print("\n--- Example: Accessing the first sequence from the LIST ---")
            first_sequence = sequence_list[0]
            print(f"  Sequence (first 60 chars): {first_sequence[:60]}...")
            print(f"  Its length is: {len(first_sequence)} amino acids.")

: 

In [ ]:
# Cell 2: Protein Embedding Generation and File Output (Corrected Version)

import torch
import transformers
import numpy as np
import re
import json

# --- Check if the input data from the previous cell exists ---
try:
    fasta_results
except NameError:
    # A helpful error message if the previous script wasn't run
    raise NameError("The 'fasta_results' dictionary was not found. Please run the previous cell to load the data from files.")

# --- Constants for the ESM2 Model and Output File ---
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
OUTPUT_FILENAME = "protein_embeddings.json"

# --- MODIFIED: The parse_fasta_string function is no longer needed ---
# The new fasta_results dictionary is already parsed into ID -> Sequence.

def get_protein_embedding(sequence: str, model, tokenizer, device) -> np.ndarray | None:
    """Generates a mean-pooled embedding for a protein sequence string using an ESM2 model."""
    # This function is already correct and needs no changes.
    sequence = re.sub(r"[UZOB]", "X", sequence) # Replace non-standard amino acids
    if not sequence: return None
    
    # Truncation is important for long sequences
    inputs = tokenizer(sequence, return_tensors='pt', truncation=True, max_length=1022)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    with torch.no_grad():
        output = model(**inputs)
    
    # Mean pooling over the sequence (ignore start/end tokens)
    hidden_states = output.last_hidden_state
    embedding = hidden_states[0, 1:-1].mean(dim=0)
    return embedding.cpu().numpy()

# --- Main Logic for Generating and Saving Embeddings ---

# 1. LOAD THE ESM2 MODEL (this might take a minute on the first run)
print(f"--- Loading Protein Language Model ({MODEL_NAME}) ---")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

# 2. GENERATE EMBEDDINGS
print("\n--- Generating Protein Embeddings ---")
# --- MODIFIED: This dictionary is now a simple map of Accession ID -> vector ---
all_embeddings = {}

# --- MODIFIED: The loop is now much simpler ---
for accession_id, sequence in fasta_results.items():
    if not sequence:
        print(f"  Skipping {accession_id}, no sequence data found.")
        continue

    print(f"  Processing: {accession_id}...")
    
    # Generate the embedding using the sequence
    embedding_vector = get_protein_embedding(sequence, model, tokenizer, device)
    
    # Store the embedding using its accession ID as the unique key
    if embedding_vector is not None:
        all_embeddings[accession_id] = embedding_vector

# 3. SAVE EMBEDDINGS TO A FILE
print(f"\n--- Saving Embeddings to File ---")

# --- MODIFIED: Simplified structure for JSON serialization ---
# Create a new dictionary to store embeddings as lists for JSON
embeddings_for_json = {}
for accession_id, vector in all_embeddings.items():
    # JSON cannot handle NumPy arrays, so convert them to standard Python lists
    embeddings_for_json[accession_id] = vector.tolist()

try:
    with open(OUTPUT_FILENAME, 'w') as f:
        json.dump(embeddings_for_json, f, indent=4)
    print(f"✅ Embeddings successfully saved to '{OUTPUT_FILENAME}'")
except IOError as e:
    print(f"❌ Error: Could not write to file '{OUTPUT_FILENAME}'. Reason: {e}")

# 4. DISPLAY RESULTS
print("\n\n--- Embedding Generation Complete: Summary ---")
total_embedded = len(all_embeddings)

if total_embedded > 0:
    print(f"Successfully generated embeddings for {total_embedded} proteins.")
    
    # --- MODIFIED: Display a sample from the new, simpler data structure ---
    # Display details for the first protein as a sample
    first_id = next(iter(all_embeddings.keys()))
    first_vector = all_embeddings[first_id]
    
    print("\n#=======================================")
    print(f"# Sample Embedding Details")
    print("#=======================================")
    print(f"  Accession ID: {first_id}")
    print(f"    - Embedding Shape: {first_vector.shape}")
    print(f"    - Embedding Sample (first 5 values): {first_vector[:5]}")
else:
    print("No embeddings were generated. Please check your input data.")